# A Dose-Response Evaluation of Explanation Metrics under Controlled Shortcut Reliance in CNN Image Classifiers

## Amendment A2, session 1: SD2 pilot, Pet dose -0.15 and the SD1 pilot

Pre-registered in `docs/PREDICTIONS_amendments.md` (A2). Run only after both
prediction files are uploaded to OSF or Zenodo and the placeholders are filled.

- Cell 3: SD1 pilot, ImageNet-pretrained ResNet-18 on Pet, doses 0 and -0.5,
  seed 7. The first printed model line gives the minutes per pretrained model.
- Cell 4: SD2 CIFAR-10 ResNet-18 pilot, dose -0.05, seed 42 (training only).
- Cells 5-6: Pet from scratch at dose -0.15, seeds 42 1 2, trained and scored.
- Cell 7: pilot report (reliance, accuracies, r_c and the A2 dose set) in
  `a2_pilot_report.json`. Pilots are not scored: A2 lets only their reliance
  and accuracies be read before the dated completion notes are committed.
- Cell 8: `a2_session1_results.zip` in the notebook's Output.

Before running: **Settings → Accelerator → GPU T4 x2**, and **Internet on**
(Captum and the ImageNet weights). Then **Save Version → Save & Run All (Commit)**.

In [ ]:
# 1. Copy the uploaded project into the writable working folder
import glob, os, shutil, subprocess, sys
found = glob.glob('/kaggle/input/**/dose_response.py', recursive=True)
assert found, 'Project files not found: add your uploaded dataset with "+ Add Input".'
src = os.path.dirname(found[0])
WORK = '/kaggle/working/project'
if not os.path.exists(WORK):
    shutil.copytree(src, WORK)
os.chdir(WORK)
print('project copied from', src)
print(sorted(os.listdir(WORK)))

In [ ]:
# 2. Install the one missing package, check the GPU and the data, get the ImageNet weights
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'captum==0.9.0'], check=True)
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU: turn on Settings → Accelerator → GPU, then restart.'
for name in ('trainval.npz', 'test.npz'):
    assert os.path.exists(f'data/pet128/{name}'), f'data/pet128/{name} missing: rebuild the bundle with -Dataset a2.'
assert os.path.isdir('data/cifar-10-batches-py'), 'data/cifar-10-batches-py missing: rebuild the bundle with -Dataset a2.'
print('Pet and CIFAR-10 data found')
from torchvision.models import ResNet18_Weights, resnet18
try:
    resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)  # cached for the training cells
except Exception as error:
    raise RuntimeError('ImageNet weights download failed: turn Internet on (Settings → Internet).') from error
print('ImageNet ResNet-18 weights ready')

In [ ]:
# 3. SD1 pilot: ImageNet-pretrained ResNet-18 on Pet, doses 0 and -0.5, seed 7.
#    The first model line (pet_resnet18_pretrained_patch_dose+000_seed7) gives the minutes per model.
!python -u -W ignore dose_response.py train --bases pet_resnet18_pretrained --shortcuts patch --doses 0 -0.5 --seeds 7

In [ ]:
# 4. SD2 pilot: CIFAR-10 ResNet-18, dose -0.05, seed 42 (training only; not scored until the A2 note)
!python -u -W ignore dose_response.py train --bases cifar10_resnet18 --shortcuts patch --doses -0.05 --seeds 42

In [ ]:
# 5. SD2: Pet from scratch at dose -0.15, seeds 42 1 2
!python -u -W ignore dose_response.py train --bases pet_resnet18 --shortcuts patch --doses -0.15 --seeds 42 1 2

In [ ]:
# 6. Score the Pet dose -0.15 models (metrics, oracle maps, controls, behaviour labels).
#    The pilots have no scores, so placebo skips them.
!python -u -W ignore dose_response.py evaluate --bases pet_resnet18 --shortcuts patch --doses -0.15 --seeds 42 1 2
!python -u -W ignore dose_response.py placebo --bases pet_resnet18 --shortcuts patch --doses -0.15 --seeds 42 1 2
!python -u -W ignore dose_response.py behaviour --selection-only --bases pet_resnet18 --shortcuts patch --doses -0.15 --seeds 42 1 2

In [ ]:
# 7. Pilot report: reliance, accuracies, r_c and the A2 dose set (written to a2_pilot_report.json)
import json
sys.path.insert(0, WORK)
import dose_response
from xai.data import num_classes
from xai.stats import chance_floor, corrected_reliance
PILOTS = [('pet_resnet18_pretrained', 0.0, 7), ('pet_resnet18_pretrained', -0.5, 7), ('cifar10_resnet18', -0.05, 42)]
report = []
for base, dose, seed in PILOTS:
    name = dose_response.model_name(base, 'patch', dose, seed)
    m = json.load(open(f'checkpoint/dose/{name}_metrics.json'))
    floor = float(chance_floor(m['test_accuracy_no_shortcut'], num_classes(base)))
    rc = float(corrected_reliance(m['reliance'], floor))
    entry = {'model': name, 'reliance': m['reliance'], 'test_accuracy': m['test_accuracy'],
             'test_accuracy_no_shortcut': m['test_accuracy_no_shortcut'], 'chance_floor': round(floor, 2),
             'reliance_corrected': round(rc, 2), 'training_minutes': m['training_minutes']}
    # The A2 rules use the CIFAR pilot (dose -0.05) and the SD1 pilot at dose 0.
    if dose != -0.5:
        entry['a2_dose_set'] = dose_response.a2_dose_set(base, rc)
    report.append(entry)
    print(entry)
json.dump(report, open('a2_pilot_report.json', 'w'), indent=2)

In [ ]:
# 8. Pack the results to download (explicit name patterns: the prefix
#    pet_resnet18_ alone would also match pet_resnet18_pretrained_)
import zipfile
out = '/kaggle/working/a2_session1_results.zip'
patterns = ['checkpoint/dose/pet_resnet18_pretrained_patch_*',
            'checkpoint/dose/cifar10_resnet18_patch_dose-005_seed42*',
            'checkpoint/dose/pet_resnet18_patch_dose-015_*',
            'results/dose_response/per_model/pet_resnet18_patch_dose-015_*',
            'a2_pilot_report.json']
files = sorted({f for p in patterns for f in glob.glob(p)})
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, f)  # keeps checkpoint/... and results/... paths
print(len(files), 'files ->', out)
shutil.rmtree(WORK)  # keep only the zip in the Output